# Project 16 — Local Course Tutor
## QA Over Lecture Notes and Slides

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Setup and Sample Lecture Material

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from pathlib import Path

llm = ChatOllama(model="qwen3:8b", temperature=0.2)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

lectures = [
    Document(page_content="""Lecture 1: Introduction to Databases
A database is an organized collection of structured data. DBMS (Database
Management System) provides tools to create, read, update, delete data.

Types: Relational (SQL), Document (MongoDB), Key-Value (Redis), Graph (Neo4j).

ACID Properties:
- Atomicity: Transactions are all-or-nothing
- Consistency: Data remains valid after transactions
- Isolation: Concurrent transactions don't interfere
- Durability: Committed data survives failures""",
        metadata={"lecture": 1, "topic": "intro", "week": 1}),

    Document(page_content="""Lecture 2: SQL Fundamentals
SQL operations: SELECT, INSERT, UPDATE, DELETE.

Joins: INNER JOIN returns matching rows. LEFT JOIN includes all left rows.
FULL OUTER JOIN includes all rows from both tables.

Aggregation: GROUP BY with COUNT, SUM, AVG, MAX, MIN.
HAVING filters groups (vs WHERE which filters rows).

Subqueries: Nested SELECT statements. Correlated subqueries reference outer query.""",
        metadata={"lecture": 2, "topic": "sql", "week": 2}),

    Document(page_content="""Lecture 3: Normalization
Normalization reduces data redundancy.
1NF: Eliminate repeating groups, atomic values only.
2NF: 1NF + no partial dependencies on composite keys.
3NF: 2NF + no transitive dependencies.
BCNF: Every determinant is a candidate key.

Denormalization: Intentionally adding redundancy for read performance.
Common in data warehouses and analytics workloads.""",
        metadata={"lecture": 3, "topic": "normalization", "week": 3}),
]
print(f"Loaded {len(lectures)} lectures")

## Step 2 — Build Course Index

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(lectures)

vectorstore = Chroma.from_documents(chunks, embeddings,
    persist_directory="sample_data/course_chroma", collection_name="db_course")
print(f"Course index: {len(chunks)} chunks from {len(lectures)} lectures")

## Step 3 — Tutor Q&A

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

tutor_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a patient CS tutor helping a student study for exams.
Use the lecture material to answer. If the concept is complex:
1. Start with a simple analogy
2. Give the technical explanation
3. Provide an example
Reference which lecture the information comes from.

Lecture Material:
{context}

Student Question: {question}

Tutor:"""
)

tutor = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": tutor_prompt},
)

student_questions = [
    "What's the difference between INNER JOIN and LEFT JOIN?",
    "Explain ACID properties with real-world examples",
    "What is 3NF and why does it matter?",
    "When would you choose MongoDB over PostgreSQL?",
]

for q in student_questions:
    print(f"\nStudent: {q}")
    result = tutor.invoke({"query": q})
    print(f"Tutor: {result['result']}")

## Step 4 — Generate Study Guide

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

guide_prompt = ChatPromptTemplate.from_messages([
    ("system", """Generate an exam study guide from the lecture material.
Include: key terms with definitions, important formulas/rules,
common exam questions, and tips for remembering concepts."""),
    ("human", "Lectures:\n{lectures}")
])

guide_chain = guide_prompt | llm | StrOutputParser()
all_lectures = "\n\n---\n\n".join([l.page_content for l in lectures])
study_guide = guide_chain.invoke({"lectures": all_lectures})
print("EXAM STUDY GUIDE:")
print(study_guide)

## What You Learned
- **Lecture content indexing** with week/topic metadata
- **Pedagogical prompting** (analogy → technical → example)
- **Study guide generation** from course material